# Sports RAG — Data Collection & Chunking

Install dependencies

In [4]:
!pip install wikipedia-api wikipedia langchain langchain-text-splitters \
             sentence-transformers pinecone tqdm

  Using cached wikipedia_api-0.11.0-py3-none-any.whl.metadata (28 kB)
  Using cached wikipedia-1.4.0.tar.gz (27 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached langchain-1.2.13-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_text_splitters-1.1.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
  Using cached pinecone-8.1.0-py3-none-any.whl.metadata (14 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached anyio-4.13.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.

  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run successfully.
  │ exit code: 1
  ╰─> [30 lines of output]
        Using cached maturin-1.12.6.tar.gz (269 kB)
        Installing build dependencies: started
        Installing build dependencies: finished with status 'done'
        Getting requirements to build wheel: started
        Getting requirements to build wheel: finished with status 'done'
        Installing backend dependencies: started
        Installing backend dependencies: finished with status 'done'
        Preparing metadata (pyproject.toml): started
        Preparing metadata (pyproject.toml): finished with status 'error'
        error: subprocess-exited-with-error
      
        Ã— Preparing metadata (pyproject.toml) did not run successfully.
        â”‚ exit code: 1
        â•°â”€> [5 lines of output]
            C:\Users\Suffi\AppData\Local\Temp\pip-build-env-pqj4a1vi\overlay\lib\python3.11\site-packages\setuptools\_

## Cell 2 — Imports

In [5]:
import wikipedia
import json
import re
import time
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

wikipedia.set_lang("en")
print("All imports OK")

c:\Users\Suffi\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


## Cell 3 — Topic list (edit freely)
These cover football, cricket, basketball, tennis and general sports history.
Feel free to add or remove topics — aim for 60+ to guarantee 500+ chunks.

In [6]:
TOPICS = [
    # --- Football / Soccer ---
    "FIFA World Cup",
    "2022 FIFA World Cup",
    "2018 FIFA World Cup",
    "UEFA Champions League",
    "Premier League",
    "La Liga",
    "Serie A",
    "Bundesliga",
    "Lionel Messi",
    "Cristiano Ronaldo",
    "Neymar",
    "Kylian Mbappe",
    "Real Madrid CF",
    "FC Barcelona",
    "Manchester United F.C.",
    "Liverpool F.C.",
    "Bayern Munich",
    "Paris Saint-Germain F.C.",
    "FIFA Ballon d'Or",
    "UEFA European Championship",
    "Copa America",
    "AFC Asian Cup",

    # --- Cricket ---
    "Cricket World Cup",
    "2023 Cricket World Cup",
    "ICC T20 World Cup",
    "Indian Premier League",
    "Test cricket",
    "Sachin Tendulkar",
    "Virat Kohli",
    "Babar Azam",
    "Shaheen Shah Afridi",
    "Pakistan national cricket team",
    "India national cricket team",
    "Australia national cricket team",
    "England cricket team",
    "The Ashes",
    "Wasim Akram",
    "Imran Khan",

    # --- Basketball ---
    "NBA",
    "NBA Finals",
    "LeBron James",
    "Michael Jordan",
    "Stephen Curry",
    "Los Angeles Lakers",
    "Golden State Warriors",
    "Chicago Bulls",
    "NBA All-Star Game",

    # --- Tennis ---
    "Wimbledon Championships",
    "US Open (tennis)",
    "French Open",
    "Australian Open",
    "Roger Federer",
    "Rafael Nadal",
    "Novak Djokovic",
    "Serena Williams",
    "Grand Slam (tennis)",

    # --- Olympics & General ---
    "Summer Olympic Games",
    "Winter Olympic Games",
    "2024 Summer Olympics",
    "Olympic Games",
    "FIFA",
    "International Cricket Council",
    "History of association football",
    "Pele",
    "Diego Maradona",
    "Usain Bolt",
    "Muhammad Ali",
]

print(f"Total topics defined: {len(TOPICS)}")

Total topics defined: 67


## Cell 4 — Fetch Wikipedia articles
This may take 3-5 minutes. Failed topics are skipped automatically.

In [7]:
def clean_text(text: str) -> str:
    """Remove citation markers, excess whitespace, and short lines."""
    text = re.sub(r"=={1,3}.*?=={1,3}", "", text)   # section headers
    text = re.sub(r"\[\d+\]", "", text)              # [1] [2] citations
    text = re.sub(r"\n{3,}", "\n\n", text)           # triple+ newlines
    text = re.sub(r"[ \t]+", " ", text)              # multiple spaces
    return text.strip()


raw_docs = []
failed  = []

for topic in tqdm(TOPICS, desc="Fetching articles"):
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        content = clean_text(page.content)
        if len(content) > 300:          # skip stubs
            raw_docs.append({
                "title"  : page.title,
                "url"    : page.url,
                "content": content
            })
    except wikipedia.exceptions.DisambiguationError as e:
        # try the first suggested option
        try:
            page = wikipedia.page(e.options[0], auto_suggest=False)
            content = clean_text(page.content)
            raw_docs.append({"title": page.title, "url": page.url, "content": content})
        except:
            failed.append(topic)
    except Exception as ex:
        failed.append(topic)
    time.sleep(0.3)   # be polite to Wikipedia

print(f"\nSuccessfully fetched : {len(raw_docs)} articles")
print(f"Failed / skipped     : {len(failed)} -> {failed}")

# Save raw backup
with open("corpus_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw_docs, f, ensure_ascii=False, indent=2)
print("corpus_raw.json saved.")

Fetching articles: 100%|██████████| 67/67 [03:32<00:00,  3.18s/it]


Successfully fetched : 67 articles
Failed / skipped     : 0 -> []
corpus_raw.json saved.


## Cell 5 — Chunk the articles (two strategies)
We create **both** Fixed and Recursive chunks for the ablation study.
The Recursive chunks are the ones you will embed and use in production.

In [8]:
# --- Strategy A: Fixed-size chunking ---
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 500,
    chunk_overlap = 0,       # no overlap = fixed
    length_function = len,
)

# --- Strategy B: Recursive chunking (production) ---
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 500,
    chunk_overlap = 50,
    length_function = len,
    separators    = ["\n\n", "\n", ". ", " ", ""],
)


def make_chunks(docs, splitter, strategy_name):
    chunks = []
    for doc in docs:
        parts = splitter.split_text(doc["content"])
        for i, part in enumerate(parts):
            part = part.strip()
            if len(part) < 50:   # skip tiny leftover fragments
                continue
            chunks.append({
                "id"      : f"{strategy_name}_{doc['title'].replace(' ','_')}_{i}",
                "text"    : part,
                "source"  : doc["title"],
                "url"     : doc["url"],
                "strategy": strategy_name,
                "chunk_no": i
            })
    return chunks


fixed_chunks     = make_chunks(raw_docs, fixed_splitter,     "fixed")
recursive_chunks = make_chunks(raw_docs, recursive_splitter, "recursive")

print(f"Fixed chunks     : {len(fixed_chunks)}")
print(f"Recursive chunks : {len(recursive_chunks)}")

# Save both for ablation comparison
with open("chunks_fixed.json", "w", encoding="utf-8") as f:
    json.dump(fixed_chunks, f, ensure_ascii=False, indent=2)

with open("chunks_recursive.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks, f, ensure_ascii=False, indent=2)

# This is the one your app will use
with open("chunks.json", "w", encoding="utf-8") as f:
    json.dump(recursive_chunks, f, ensure_ascii=False, indent=2)

print("\nSaved: chunks.json, chunks_fixed.json, chunks_recursive.json")

Fixed chunks     : 8471
Recursive chunks : 9261

Saved: chunks.json, chunks_fixed.json, chunks_recursive.json


## Cell 6 — Quick sanity check

In [10]:
import json

with open("chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks loaded : {len(chunks)}")
print(f"Avg chunk length    : {sum(len(c['text']) for c in chunks) // len(chunks)} chars")
print(f"Unique sources      : {len(set(c['source'] for c in chunks))}")
print()
print("--- Sample chunk ---")
sample = chunks[50]
print(f"ID     : {sample['id']}")
print(f"Source : {sample['source']}")
print(f"Text   : {sample['text'][:300]}...")

Total chunks loaded : 9261
Avg chunk length    : 343 chars
Unique sources      : 67

--- Sample chunk ---
ID     : recursive_FIFA_World_Cup_50
Source : FIFA World Cup
Text   : The base contains two layers of semi-precious malachite while the bottom side of the trophy bears the engraved year and name of each FIFA World Cup winner since 1974. The description of the trophy by Gazzaniga was: "The lines spring out from the base, rising in spirals, stretching out to receive the...


## Cell 7 — Embed chunks & upsert to Pinecone


In [ ]:
PINECONE_API_KEY = "" #<--- INSERT YOUR PINECONE API KEY HERE   
INDEX_NAME       = "sports-rag"
EMBEDDING_DIM    = 384                        # all-MiniLM-L6-v2 dimension
BATCH_SIZE       = 100

# Load chunks
with open("chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)

# Load embedding model
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create index if it doesn't exist
existing_indexes = [idx.name for idx in pc.list_indexes()]
if INDEX_NAME not in existing_indexes:
    pc.create_index(
        name      = INDEX_NAME,
        dimension = EMBEDDING_DIM,
        metric    = "cosine",
        spec      = ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"Index '{INDEX_NAME}' created.")
else:
    print(f"Index '{INDEX_NAME}' already exists — upserting into it.")

index = pc.Index(INDEX_NAME)

# Embed and upsert in batches
print(f"\nEmbedding {len(chunks)} chunks in batches of {BATCH_SIZE}...")
for i in tqdm(range(0, len(chunks), BATCH_SIZE), desc="Upserting"):
    batch = chunks[i : i + BATCH_SIZE]
    texts = [c["text"] for c in batch]

    embeddings = model.encode(texts, show_progress_bar=False).tolist()

    vectors = [
        (
            c["id"],
            emb,
            {"text": c["text"], "source": c["source"], "url": c["url"]}
        )
        for c, emb in zip(batch, embeddings)
    ]
    index.upsert(vectors=vectors)

print("\nAll chunks embedded and upserted to Pinecone!")
stats = index.describe_index_stats()
print(f"Total vectors in index: {stats['total_vector_count']}")

Loading embedding model...


c:\Users\Suffi\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Suffi\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8790.94it/s]
BertMo

Model loaded.
Index 'sports-rag' created.

Embedding 9261 chunks in batches of 100...


Upserting:  17%|█▋        | 16/93 [00:49<03:56,  3.07s/it]


PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 25 Mar 2026 14:43:46 GMT', 'Content-Type': 'application/json', 'Content-Length': '97', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '455', 'x-envoy-upstream-service-time': '29', 'x-pinecone-response-duration-ms': '458', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"Vector ID must be ASCII, but got 'recursive_Kylian_Mbappé_0'","details":[]}


## Cell 8 — Verify Pinecone with a test query

In [12]:
test_query = "Who won the FIFA World Cup in 2022?"

query_vec = model.encode(test_query).tolist()
results   = index.query(vector=query_vec, top_k=3, include_metadata=True)

print(f"Query: {test_query}\n")
for i, match in enumerate(results["matches"]):
    print(f"--- Result {i+1} (score: {match['score']:.4f}) ---")
    print(f"Source : {match['metadata']['source']}")
    print(f"Text   : {match['metadata']['text'][:200]}...")
    print()

Query: Who won the FIFA World Cup in 2022?

--- Result 1 (score: 0.7155) ---
Source : 2022 FIFA World Cup
Text   : The 2022 FIFA World Cup was the 22nd FIFA World Cup, the quadrennial world championship for national football teams organised by FIFA. It took place in Qatar from 20 November to 18 December 2022, afte...

--- Result 2 (score: 0.6237) ---
Source : FIFA World Cup
Text   : . The reigning champions are Argentina, who won their third title at the 2022 World Cup by defeating France....

--- Result 3 (score: 0.5776) ---
Source : Lionel Messi
Text   : . At the 2022 FIFA World Cup, Messi led Argentina to its first World Cup victory in 36 years, scoring twice in the final to defeat France. Having scored seven goals and assisted three during the tourn...



## Done!
Your output files:
| File | Purpose |
|---|---|
| `corpus_raw.json` | Raw Wikipedia articles (backup) |
| `chunks.json` | Recursive chunks → upload to HF Space |
| `chunks_fixed.json` | Fixed chunks → for ablation study |
| `chunks_recursive.json` | Same as chunks.json (explicit copy) |

**Next step:** Upload `chunks.json` to your HF Space repo and build the BM25 + semantic search pipeline.